# Figures S8 and S12 — the 100 LHS conditions: simplex coverage and PCA of the endpoint spectra

* S8: `figS08.py` (original `10_eda_S14.ipynb`, `fig5_coverage`), no title in the SI version.
* S12: `figS12.py` (original `PCA/pca_decomposition.ipynb`): 6-component PCA, max-normalised, fitted on the 88 non-flat endpoint spectra (std < 0.015 over 310–1000 nm). λmax = raw argmax of the endpoint spectrum.

⚑ The frozen basis actually used by the PCA-cost BO runs (`pca_model_norm6.npz`, 2026-06-22) is **not** this fit (max |Δmean| 0.049) and could not be regenerated from the 100 spectra.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
import sys
import types

from IPython.core.magic import register_cell_magic

MODULE_SOURCE = {}


@register_cell_magic
def module(line, cell):
    """`%%module a.b`: run this cell as the Python module a.b, so that the original code keeps its own namespaces."""
    name = line.strip()
    parts = name.split(".")
    for i in range(1, len(parts)):                              # parent packages
        pkg = ".".join(parts[:i])
        if pkg not in sys.modules:
            sys.modules[pkg] = types.ModuleType(pkg)
            sys.modules[pkg].__path__ = []
            if i > 1:
                setattr(sys.modules[".".join(parts[:i - 1])], parts[i - 1], sys.modules[pkg])
    mod = types.ModuleType(name)
    mod.__package__ = name.rpartition(".")[0]
    sys.modules[name] = mod
    setattr(sys.modules[mod.__package__], parts[-1], mod)
    MODULE_SOURCE[name] = cell
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


In [ ]:
%%module toofast_analysis.paths
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


In [ ]:
from toofast_analysis.paths import *

### `lhs100`

In [ ]:
"""The 100 projected-LHS conditions (6 RG campaigns, AvaSpec-2048-USB2) behind Fig 3, S8 and S12.

Port of 2FAST_ML/10_eda_S14.ipynb (loading, convergence, flat flag, spectral area, logistic fits) and
2FAST_ML/14_descriptor_correlation.ipynb (descriptor table). Code and parameters are kept as in those notebooks.

NOTE (decision 2026-10-12, option a): the two original notebooks recompute the "growth time" with DIFFERENT
settings, and both differ from the platform's own termination rule. Both are kept so the figures reproduce:
  - SETTINGS_NB10 (Fig 3-I colours, flat points, 3-III):  Δ|Ext_max| < 1.5 %, Pearson > 0.998, ≥ 4 spectra, flat if std < 0.015
  - SETTINGS_NB14 (Fig 3-II Pearson matrix):             Δ|Ext_max| < 2 %,   Pearson > 0.998, ≥ 10 spectra, flat if std < 0.03
In both, consecutive raw spectra (n vs n-1) are compared.

DECISION 2026-10-22 (Fig 3-II): the descriptor matrix is shown as Spearman ρ (spearman_matrix) on the nb14 clean set
minus BASELINE_SHIFT, for Ext_max, area, λmax and the plateau-onset growth time (fig03.SPEARMAN_DESCRIPTORS; the
nb14 growth time floors half of the runs at the 10th spectrum). The original Pearson matrix is kept (fig03 result
"pearson") for the record.
"""
import csv
import re

import h5py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import find_peaks, savgol_filter
from scipy.stats import spearmanr


REAGENTS = ["AgNO3", "PVP", "AcOH", "AscAcid", "NaBH4"]
QCOLS = ["Q1", "Q2", "Q3", "Q4", "Q5"]
STOCK = np.array([1.8, 1.2, 1256.0, 1.8, 0.036])   # stock concentrations, mM (PVP per chain)
STANDARD_FLOW, STANDARD_TOL = 2.8, 0.15

SETTINGS_NB10 = dict(amp=0.015, corr=0.998, kmin=4, flat_std=0.015)
SETTINGS_NB14 = dict(amp=0.02, corr=0.998, kmin=10, flat_std=0.03)

EVOL_WL_RANGE = (315, 950)   # nm, spectral-evolution panels and spectral area of the kinetics (nb10)
SG_WINDOW, SG_POLY = 15, 3


# ── loading ──────────────────────────────────────────────────────────────────
def file_list():
    with open(RESOURCE_DIR / "lhs100_files.csv", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def _classify(filename, flows):
    stem = filename.lower()
    at_std = np.abs(np.asarray(flows) - STANDARD_FLOW) <= STANDARD_TOL
    if "_iter_" in stem:
        return "BO_ITER"
    if "_init_" in stem:
        return "BO_INIT"
    if "_sample_" in stem:
        return "STANDARD" if at_std.all() else "LHS"
    return "UNKNOWN"


def _manip_of(filename):
    return re.split(r"_(?:sample|init|iter)_", filename)[0]


def load(raw=None):
    """Return (df, spectra): one row per H5 (sorted by manip, timestamp) and the time series keyed by filename."""
    raw = RAW if raw is None else raw
    rows, spectra = [], {}
    for item in file_list():
        with h5py.File(raw / item["raw_path"], "r") as f:
            flows = f["conditions/flow_set"][()]
            lam = f["spectra/lambda_max"][()]
            amx = f["spectra/extinction_max"][()]
            ts = [t.decode() if isinstance(t, bytes) else t for t in f["spectra/timestamp"][()]]
            wl = f["spectra/wavelength"][()]
            absb = f["spectra/extinction"][()]
            meta = dict(f["sample"].attrs)
            stat = dict(f["result"].attrs)
        name = item["filename"]
        try:
            tsec = (pd.to_datetime(ts) - pd.to_datetime(ts[0])).total_seconds().values
        except Exception:
            tsec = np.arange(len(lam), dtype=float)
        rows.append({
            "filename": name, "manip": _manip_of(name), "sample_id": meta.get("sample_id", ""),
            "timestamp": meta.get("timestamp", ""), "exp_type": _classify(name, flows),
            "Q1": flows[0], "Q2": flows[1], "Q3": flows[2], "Q4": flows[3], "Q5": flows[4],
            "Q_total": float(np.sum(flows)),
            "lambda_max": float(stat.get("lambda_max_nm", lam[-1])),
            "abs_max": float(stat.get("extinction_max", amx[-1])),
            "lmax_initial": float(lam[0]), "abs_initial": float(amx[0]),
            "n_spectra": len(lam), "growth_time_s": float(tsec[-1]),
        })
        spectra[name] = {"wl": wl, "abs": absb, "lam_ts": lam, "abs_ts": amx, "t": tsec}
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values(["manip", "timestamp"]).reset_index(drop=True)
    df["seq"] = np.arange(len(df))
    return df, spectra


def add_chem_features(df):
    mol = df[QCOLS].values * STOCK
    ag = mol[:, 0]
    df["reductant_to_Ag"] = (mol[:, 3] + mol[:, 4]) / ag
    df["NaBH4_to_Ag"] = mol[:, 4] / ag
    df["nucleation_vs_growth"] = mol[:, 4] / np.where(mol[:, 3] == 0, np.nan, mol[:, 3])
    df["PVP_to_Ag"] = mol[:, 1] / ag
    df["acid_to_Ag"] = mol[:, 2] / ag
    return df


# ── convergence (recomputed by the analysis, see module docstring) ─────────────
def convergence_time(sp, amp, corr, kmin):
    """First elapsed time where consecutive spectra satisfy both criteria -> (time_s, index, status)."""
    A = np.asarray(sp["abs"]); amx = np.asarray(sp["abs_ts"]); t = np.asarray(sp["t"])
    n = len(t)
    if n < kmin:
        return np.nan, n - 1, "too_short"
    for i in range(1, n):
        if (i + 1) < kmin:
            continue
        d_amp = abs(amx[i] - amx[i - 1]) / max(abs(amx[i - 1]), 1e-9)
        r = np.corrcoef(A[i], A[i - 1])[0, 1]
        if d_amp < amp and r > corr:
            return float(t[i]), i, "ok"
    return np.nan, n - 1, "never"


def enrich_convergence(df, spectra, settings):
    s = settings
    ct, idx, st = zip(*(convergence_time(spectra[f], s["amp"], s["corr"], s["kmin"]) for f in df.filename))
    df = df.copy()
    df["conv_time_s"], df["conv_index"], df["conv_status"] = ct, idx, st
    df["abs_std"] = [float(np.asarray(spectra[f]["abs"][-1]).std()) for f in df.filename]
    df["is_flat"] = df.abs_std < s["flat_std"]
    return df


def table(settings, raw=None):
    """Loaded set with chemistry features and the convergence / flat columns for the given settings."""
    df, spectra = load(raw)
    return enrich_convergence(add_chem_features(df), spectra, settings), spectra


# ── kinetics helpers (nb10) ──────────────────────────────────────────────────
def spectral_area_series(sp, wl_range=EVOL_WL_RANGE):
    """Trapezoidal area of every spectrum over wl_range (no clipping) — used for the growth rates."""
    wl = np.asarray(sp["wl"]); A = np.asarray(sp["abs"])
    m = (wl >= wl_range[0]) & (wl <= wl_range[1])
    return np.trapz(A[:, m], wl[m], axis=1)


def logistic(t, y0, ymax, k, t0):
    return y0 + (ymax - y0) / (1 + np.exp(-k * (t - t0)))


def fit_logistic(t, y):
    """Bounded 4-parameter logistic fit -> (t_smooth, y_smooth, popt) or (None, None, None)."""
    spread = max(y.max() - y.min(), 1e-6)
    pad = 0.25 * spread
    t_span = max(t[-1] - t[0], 1e-6)
    p0 = [y[0], y[-1], 4 / t_span, t[len(t) // 2]]
    lo = [y.min() - pad, y.min() - pad, 1e-4, t.min()]
    hi = [y.max() + pad, y.max() + pad, 30 / t_span, t.max()]
    try:
        popt, _ = curve_fit(logistic, t, y, p0=p0, bounds=(lo, hi), maxfev=20000)
    except RuntimeError:
        return None, None, None
    t_smooth = np.linspace(t.min(), t.max(), 200)
    return t_smooth, logistic(t_smooth, *popt), popt


def logistic_max_slope(popt):
    """Slope at the inflection point of the fitted logistic (the 'max rate')."""
    y0, ymax, k, _ = popt
    return k * (ymax - y0) / 4


def kinetics(sp):
    """Time-sorted Ext_max(t), spectral area(t), their logistic fits and max rates (Fig 3-III c, d)."""
    t = np.asarray(sp["t"]); order = np.argsort(t); t = t[order]
    amax = np.asarray(sp["abs_ts"])[order]
    area = spectral_area_series(sp)[order]
    fit_amax, fit_area = fit_logistic(t, amax), fit_logistic(t, area)
    rates = None
    if fit_amax[2] is not None and fit_area[2] is not None:
        rates = dict(ext_max=logistic_max_slope(fit_amax[2]), area=logistic_max_slope(fit_area[2]))
    return dict(t=t, ext_max=amax, area=area, fit_ext_max=fit_amax, fit_area=fit_area, rates=rates)


# ── descriptors (nb14) ───────────────────────────────────────────────────────
SECOND_PEAK_CENTER, SECOND_PEAK_HALFWIN = 335.0, 15.0
SECOND_PEAK_CTX_HALF, SECOND_PEAK_PROMINENCE = 25.0, 0.005
DESCRIPTORS = ["Ext_max", "area", "secondary_peak_335", "lambda_max", "peak_ratio", "growth_time_s"]
DESC_LABELS = {"Ext_max": r"$Ext_{\max}$", "area": "spectral area", "secondary_peak_335": r"peak @335 nm",
               "lambda_max": r"$\lambda_{\max}$ (nm)", "peak_ratio": r"$Ext_{\max}$/peak335",
               "growth_time_s": "growth time (s)"}
# Baseline shifted negative during the run (≈ −0.05 a.u. at 310 nm; spotted because their 335 nm band has an absolute
# height ≈ 0, making Ext_max/peak335 explode). The offset biases every extinction descriptor: excluded from the
# Spearman matrix (decision 2026-10-22).
BASELINE_SHIFT = ["S13-M09_sample_4", "S13-M09_sample_6", "S13-M09_sample_7"]


def _smooth(y, sg_window=SG_WINDOW, sg_poly=SG_POLY):
    n = len(y)
    win = min(sg_window, n - (n + 1) % 2)
    if win <= sg_poly:
        win = sg_poly + 1 + (sg_poly % 2)
    return savgol_filter(y, win, sg_poly)


def secondary_peak(wl, absorbance, center=SECOND_PEAK_CENTER, half_win=SECOND_PEAK_HALFWIN,
                   ctx_half=SECOND_PEAK_CTX_HALF, prominence=SECOND_PEAK_PROMINENCE):
    """Height of a genuine interior peak near 335 nm, NaN if there is none."""
    m = (wl >= center - ctx_half) & (wl <= center + ctx_half)
    if m.sum() < 9:
        return np.nan
    y = _smooth(absorbance[m]); x = wl[m]
    peaks, props = find_peaks(y, prominence=prominence)
    if len(peaks) == 0:
        return np.nan
    cand = [(p, props["prominences"][k]) for k, p in enumerate(peaks)
            if abs(x[p] - center) <= half_win and 0 < p < len(y) - 1]
    if not cand:
        return np.nan
    p, _ = max(cand, key=lambda c: c[1])
    return float(y[p])


def spectral_area_final(wl, absorbance):
    """Trapezoidal area of the baseline-clipped final spectrum over the full range — used in Fig 3-II."""
    return float(np.trapz(np.clip(absorbance, 0, None), wl))


def descriptor_table(df, spectra):
    rows = []
    for _, r in df.iterrows():
        sp = spectra[r.filename]
        wl, ab = sp["wl"], np.asarray(sp["abs"][-1])
        rows.append(dict(filename=r.filename, Ext_max=r.abs_max, lambda_max=r.lambda_max,
                         secondary_peak_335=secondary_peak(wl, ab), area=spectral_area_final(wl, ab),
                         growth_time_s=r.conv_time_s))
    d = pd.DataFrame(rows)
    d["peak_ratio"] = d.Ext_max / d.secondary_peak_335.clip(lower=1e-4)
    return d


def corr_matrix(X_df, Y_df):
    """Pearson r between every column of X_df (rows) and Y_df (cols), NaN dropped pairwise."""
    R = np.full((X_df.shape[1], Y_df.shape[1]), np.nan)
    for i, xc in enumerate(X_df.columns):
        for j, yc in enumerate(Y_df.columns):
            x, y = X_df[xc].values, Y_df[yc].values
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() > 3 and x[ok].std() > 0 and y[ok].std() > 0:
                R[i, j] = np.corrcoef(x[ok], y[ok])[0, 1]
    return R


def spearman_matrix(X_df, Y_df):
    """Spearman ρ, two-sided p-value and n between every column of X_df (rows) and Y_df (cols), NaN dropped pairwise."""
    shape = (X_df.shape[1], Y_df.shape[1])
    rho, p, n = np.full(shape, np.nan), np.full(shape, np.nan), np.zeros(shape, int)
    for i, xc in enumerate(X_df.columns):
        for j, yc in enumerate(Y_df.columns):
            x, y = X_df[xc].values, Y_df[yc].values
            ok = np.isfinite(x) & np.isfinite(y)
            n[i, j] = ok.sum()
            if ok.sum() > 3 and x[ok].std() > 0 and y[ok].std() > 0:
                rho[i, j], p[i, j] = spearmanr(x[ok], y[ok])
    return rho, p, n


### `fig03`

In [ ]:
"""Figure 3 — autonomous exploration of the synthesis space (100 projected-LHS conditions).

Panels (as assembled in manuscript v4):
  scatter   Ext_max vs λmax coloured by convergence time, flat spectra in grey, a–d ringed  (nb10 `fig8_amax_lmax_convtime`)
            → in the paper in the `plot_scatter_ms` layout (viridis, log colour scale 3–200 s, no legend)
  pearson   Pearson r between Q1–Q5 and six descriptors                              (nb14 `fig14_corr_raw_flows`)
            → replaced in the paper by `plot_spearman` (Spearman ρ, 4 descriptors, baseline-shifted runs excluded;
              decision 2026-10-22)
  a–d       time-resolved spectra of the 4 representative conditions                 (nb10 `fig8c_spectra_*`)
  c, d      Ext_max(t) and spectral area(t) with logistic fits and max rates          (nb10 `fig10e_combined_extmax_area_*`)
The regime regions (seeds / spheres / discs / others), the "No particles" line and the a–d letters were drawn
by hand on the assembled figure; they are not produced here.
Styles are those of the original notebooks; `stamp=True` reproduces their "source: …" footer exactly.
"""
import matplotlib as mpl
import numpy as np

def convergence_time_v2(A, amx, t, amp=0.02, corr=0.997, tail=3):
    """Earliest onset of the steady-state plateau (seconds), free of the 10-spectrum floor.
    Returns (time, index, status) with status in {'ok','never','too_short'}."""
    A = np.asarray(A, float); amx = np.asarray(amx, float); t = np.asarray(t, float)
    n = len(t)
    if n < 2:
        return np.nan, n - 1, "too_short"
    final = A[-1]; af = amx[-1]
    def match(i):
        d = abs(amx[i] - af) / max(abs(af), 1e-9)
        r = np.corrcoef(A[i], final)[0, 1]
        return (d < amp) and (r > corr)
    k = min(tail, n)
    if not all(match(i) for i in range(n - k, n)):     # tail is still drifting -> censored
        return np.nan, n - 1, "never"
    for i in range(n):                                 # earliest i with a stable tail from i..n
        if all(match(j) for j in range(i, n)):
            return float(t[i]), i, "ok"
    return float(t[-1]), n - 1, "ok"

REPRESENTATIVE = ["S19-M01_sample_17", "S19-M01_sample_1", "S19-M01_sample_12", "S12-M04_sample_12"]  # a, b, c, d
KINETIC_FOCUS = REPRESENTATIVE[2:4]   # c, d

PUB_STYLE = {"figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight", "font.family": "sans-serif",
             "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "font.size": 11, "axes.titlesize": 12,
             "axes.labelsize": 11, "axes.linewidth": 0.9, "axes.spines.top": False, "axes.spines.right": False,
             "xtick.direction": "out", "ytick.direction": "out", "xtick.labelsize": 10, "ytick.labelsize": 10,
             "legend.fontsize": 9, "legend.frameon": False, "lines.linewidth": 1.4, "figure.facecolor": "white",
             "axes.facecolor": "white"}
FIG8_FONTSIZE = 16
GROWTH_CMAP, GROWTH_VMAX, GROWTH_SPLIT, GROWTH_SPLIT_FRAC = "plasma", 200, 20, 0.75
# manuscript v4 scatter layout (plot_scatter_ms): viridis on a log scale, times outside [3, 200] s clipped to the ends
# fixed geometry in inches (axes w × h, colour-bar pad and width, measured on the manuscript panel) so that the plot area
# does not depend on the label lengths; the saved file is cropped to the content (savefig.bbox = "tight")
SCATTER_MS_AXES_IN, SCATTER_MS_CBAR_IN = (3.24, 2.87), (0.08, 0.16)
SCATTER_MS_FONTSIZE, SCATTER_MS_CMAP = 22, "viridis"
SCATTER_MS_TLIM, SCATTER_MS_CBAR_TICKS = (3, 200), [3, 10, 30, 100, 200]
EVOL_FIGSIZE, EVOL_CMAP, EVOL_FLOWBOX_SCALE = (5.4, 5.0), "viridis", 0.62
FIG10E_FIGSIZE = (6, 5.1)
CORR_FONTSIZE = 16
# Fig 3-II descriptors (decision 2026-10-22): peak@335 nm and Ext_max/peak335 dropped — the 335 nm band is resolved only
# for anisotropic particles and its height is mostly the underlying extinction (a proper value would need deconvolution)
# Convergence time = plateau onset, the ML-part rule (step36 convergence_time_v2: first spectrum from which every later one is
# within 2 % of the final Ext_max with Pearson > 0.997 to the final spectrum), free of the nb14 10-spectrum floor.
SPEARMAN_DESCRIPTORS = ["Ext_max", "area", "lambda_max", "growth_time_plateau_s"]
SPEARMAN_LABELS = {**DESC_LABELS, "growth_time_plateau_s": "convergence time (s)"}
SPEARMAN_ALPHA, SPEARMAN_FMT = 0.05, ".1f"     # cells with p ≥ alpha greyed; 1 decimal (SE of ρ ≈ 0.12 at n = 74)
# manuscript v4 panel layout (labels, size) — the Pearson panel as assembled in the paper
SPEARMAN_FIGSIZE = (5.7, 3.85)
SPEARMAN_MS_YLABELS = [r"$Q_{AgNO_3}$", r"$Q_{PVP}$", r"$Q_{AcOH}$", r"$Q_{AscAcid}$", r"$Q_{NaBH_4}$"]
SPEARMAN_MS_XLABELS = {"Ext_max": r"$Ext_{max}$", "area": "Spectral area", "lambda_max": r"$\lambda_{max}$",
                       "growth_time_plateau_s": "Convergence time"}


def compute(raw=None):
    """Everything Fig 3 needs: the nb10 table (panels scatter, a–d), the nb14 descriptor table and Pearson matrix,
    the Fig 3-II Spearman matrix and the c/d kinetics."""
    df10, spectra = table(SETTINGS_NB10, raw)
    df14 = enrich_convergence(df10.drop(columns=["conv_time_s", "conv_index", "conv_status", "abs_std", "is_flat"]),
                                spectra, SETTINGS_NB14)
    df14 = df14[~df14.is_flat].reset_index(drop=True)          # nb14: clean set only
    desc = descriptor_table(df14, spectra)
    desc["growth_time_plateau_s"] = [convergence_time_v2(spectra[f]["abs"], spectra[f]["abs_ts"], spectra[f]["t"])[0]
                                     for f in desc.filename]
    R = corr_matrix(df14[QCOLS], desc[DESCRIPTORS])
    keep = ~desc.filename.isin(BASELINE_SHIFT)
    rho, p, n = spearman_matrix(df14.loc[keep, QCOLS], desc.loc[keep, SPEARMAN_DESCRIPTORS])
    kin = {name: kinetics(spectra[name]) for name in KINETIC_FOCUS}
    return dict(df=df10, spectra=spectra, df_corr=df14, descriptors=desc, pearson=R, spearman=rho, spearman_p=p,
                spearman_n=n, kinetics=kin)


def save(fig, path, **kwargs):
    """Save with the notebooks' style active (ticks/text are laid out again at save time)."""
    import matplotlib.pyplot as plt
    with plt.rc_context(PUB_STYLE):
        fig.savefig(path, **kwargs)


# ── helpers ──────────────────────────────────────────────────────────────────
def _stamp(fig, source):
    fig.text(0.995, 0.003, f"source: {source}", ha="right", va="bottom", fontsize=6, color="0.55")


def _short(fn):
    return fn.replace("_sample_", "·s").replace("_init_", "·i").replace("_iter_", "·it")


def conv_norm(vmin=0, vmax=GROWTH_VMAX, split=GROWTH_SPLIT, split_frac=GROWTH_SPLIT_FRAC):
    """Colour scale expanded over [0, split] s and compressed over [split, vmax] s."""
    def fwd(x):
        x = np.asarray(x, float)
        return np.where(x <= split, split_frac * (x - vmin) / (split - vmin),
                        split_frac + (1 - split_frac) * (x - split) / (vmax - split))

    def inv(y):
        y = np.asarray(y, float)
        return np.where(y <= split_frac, vmin + y / split_frac * (split - vmin),
                        split + (y - split_frac) / (1 - split_frac) * (vmax - split))
    return mpl.colors.FuncNorm((fwd, inv), vmin=vmin, vmax=vmax)


def _frame_axes(ax, fontsize=FIG8_FONTSIZE):
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.9)
    ax.tick_params(which="both", direction="in", top=True, right=True)
    ax.tick_params(which="major", length=5.5, width=0.9, labelsize=fontsize * 0.85)
    ax.tick_params(which="minor", length=3.0, width=0.7)
    ax.xaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator())
    ax.set_box_aspect(1)


# ── panels ───────────────────────────────────────────────────────────────────
def plot_scatter(result, highlight=REPRESENTATIVE, fontsize=FIG8_FONTSIZE, stamp=False):
    import matplotlib.pyplot as plt
    d = result["df"]
    with plt.rc_context(PUB_STYLE):
        fig, ax = plt.subplots(figsize=(6.6, 4.9))
        ok, na = d[d.conv_status == "ok"], d[d.conv_status != "ok"]
        sc = ax.scatter(ok.lambda_max, ok.abs_max, c=ok.conv_time_s, cmap=GROWTH_CMAP, norm=conv_norm(),
                        s=46, edgecolor="white", linewidth=0.5, zorder=3)
        if len(na):
            ax.scatter(na.lambda_max, na.abs_max, facecolor="0.8", edgecolor="0.5", s=44, linewidth=0.5,
                       zorder=2, label=f"flat spectra (n = {len(na)})")
        if highlight:
            h = d[d.filename.isin(highlight)]
            ax.scatter(h.lambda_max, h.abs_max, s=320, facecolor="none", edgecolor="black", linewidth=1.8, zorder=5)
            ax.legend(loc="upper right", fontsize=fontsize * 0.65)
        cb = fig.colorbar(sc, ax=ax, ticks=[0, 5, 10, 15, 20, 60, 100, 150, 200])
        cb.set_label("convergence time (s)", fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize * 0.8)
        ax.set_xlabel(r"$\lambda_{\max}$ (nm)", fontsize=fontsize)
        ax.set_ylabel(r"$Ext_{\max}$ (a.u.)", fontsize=fontsize)
        ax.tick_params(labelsize=fontsize * 0.85)
        ax.set_ylim(0, float(d.abs_max.max()) * 1.12)
        fig.tight_layout()
        if stamp:
            _stamp(fig, "10_eda_S14.ipynb")
    return fig


def plot_scatter_ms(result, highlight=REPRESENTATIVE, fontsize=SCATTER_MS_FONTSIZE):
    """Scatter in the manuscript v4 layout: viridis, log colour scale, no legend (same data as plot_scatter)."""
    import matplotlib.pyplot as plt
    d = result["df"]
    vmin, vmax = SCATTER_MS_TLIM
    with plt.rc_context(PUB_STYLE):
        (aw, ah), (cpad, cw) = SCATTER_MS_AXES_IN, SCATTER_MS_CBAR_IN
        left, bottom, right, top = 1.2, 0.9, 1.2, 0.2     # room for the labels, cropped away at save time
        fw, fh = left + aw + cpad + cw + right, bottom + ah + top
        fig = plt.figure(figsize=(fw, fh))
        ax = fig.add_axes([left / fw, bottom / fh, aw / fw, ah / fh])
        cax = fig.add_axes([(left + aw + cpad) / fw, bottom / fh, cw / fw, ah / fh])
        ok, na = d[d.conv_status == "ok"], d[d.conv_status != "ok"]
        sc = ax.scatter(ok.lambda_max, ok.abs_max, c=ok.conv_time_s.clip(vmin, vmax), cmap=SCATTER_MS_CMAP,
                        norm=mpl.colors.LogNorm(vmin, vmax), s=30, edgecolor="white", linewidth=0.5, zorder=3)
        if len(na):
            ax.scatter(na.lambda_max, na.abs_max, facecolor="0.8", edgecolor="0.5", s=30, linewidth=0.5, zorder=2)
        if highlight:
            h = d[d.filename.isin(highlight)]
            ax.scatter(h.lambda_max, h.abs_max, s=200, facecolor="none", edgecolor="black", linewidth=1.5, zorder=5)
        cb = fig.colorbar(sc, cax=cax, ticks=SCATTER_MS_CBAR_TICKS)
        cb.ax.minorticks_off()
        cb.ax.set_yticklabels([f"{t:g}" for t in SCATTER_MS_CBAR_TICKS])
        cb.set_label("convergence time (s)", fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize * 0.8, length=6.5)
        ax.set_xlabel(r"$\lambda_{\max}$ (nm)", fontsize=fontsize)
        ax.set_ylabel(r"$Ext_{\max}$ (a.u.)", fontsize=fontsize)
        ax.tick_params(labelsize=fontsize * 0.8, length=6.5)
        ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(200))
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.5))
        ax.set_xlim(right=1000)
        ax.set_ylim(0, float(d.abs_max.max()) * 1.12)
    return fig


def plot_pearson(result, fontsize=CORR_FONTSIZE, stamp=False, cmap="RdBu_r", vmin=-1, vmax=1):
    """Original nb14 panel (Pearson, nb14 clean set) — superseded in the paper by plot_spearman."""
    return _corr_heatmap(result["pearson"], [DESC_LABELS[d] for d in DESCRIPTORS], "Pearson r",
                         fontsize, "14_descriptor_correlation.ipynb" if stamp else None, cmap, vmin, vmax)


def plot_spearman(result, fontsize=CORR_FONTSIZE, cmap="RdBu_r", vmin=-1, vmax=1, alpha=SPEARMAN_ALPHA,
                  fmt=SPEARMAN_FMT, figsize=SPEARMAN_FIGSIZE):
    """Fig 3-II (decision 2026-10-22): Spearman ρ on SPEARMAN_DESCRIPTORS without the baseline-shifted runs, cells
    with p ≥ alpha (two-sided, uncorrected) greyed. Layout of the manuscript v4 panel (labels, frameless cells, size);
    n and the meaning of grey go in the caption."""
    import matplotlib.pyplot as plt
    R, greyed = result["spearman"], result["spearman_p"] >= alpha
    with plt.rc_context(PUB_STYLE):
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(R, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.imshow(np.where(greyed, 1.0, np.nan), cmap=mpl.colors.ListedColormap(["0.9"]), aspect="auto")
        ax.set_xticks(range(R.shape[1]))
        ax.set_xticklabels([SPEARMAN_MS_XLABELS[d] for d in SPEARMAN_DESCRIPTORS], rotation=45, ha="right",
                           rotation_mode="anchor", fontsize=fontsize)
        ax.set_yticks(range(5))
        ax.set_yticklabels(SPEARMAN_MS_YLABELS, fontsize=fontsize)
        ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
        for i in range(R.shape[0]):
            for j in range(R.shape[1]):
                frac = abs(R[i, j] - vmin) / max(vmax - vmin, 1e-9)
                color = "0.55" if greyed[i, j] else "white" if (frac > 0.75 or frac < 0.25) else "black"
                txt = format(R[i, j], fmt)
                ax.text(j, i, "0.0" if txt == "-0.0" else txt, ha="center", va="center", fontsize=fontsize * 0.8,
                        color=color)
        cb = fig.colorbar(im, ax=ax, ticks=[-1, -0.5, 0, 0.5, 1], fraction=0.06, pad=0.04)
        cb.set_label(r"Spearman $\rho$", fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize * 0.9)
        cb.ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.1f"))
        fig.tight_layout()
    return fig


def _corr_heatmap(R, xlabels, cbar_label, fontsize, stamp_source, cmap, vmin, vmax, greyed=None, fmt=".2f",
                  note=None):
    import matplotlib.pyplot as plt
    ylabels = [f"Q{k + 1}\n{REAGENTS[k]}" for k in range(5)]
    greyed = np.zeros(R.shape, bool) if greyed is None else greyed
    with plt.rc_context(PUB_STYLE):
        fig, ax = plt.subplots(figsize=(0.95 * len(xlabels) + 2.6, 0.95 * len(ylabels) + 2.0))
        im = ax.imshow(R, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
        if greyed.any():
            ax.imshow(np.where(greyed, 1.0, np.nan), cmap=mpl.colors.ListedColormap(["0.9"]), aspect="equal")
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=40, ha="right", fontsize=fontsize)
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels, fontsize=fontsize)
        for i in range(R.shape[0]):
            for j in range(R.shape[1]):
                if np.isfinite(R[i, j]):
                    frac = abs(R[i, j] - vmin) / max(vmax - vmin, 1e-9)
                    color = "0.55" if greyed[i, j] else "white" if (frac > 0.75 or frac < 0.25) else "black"
                    txt = format(R[i, j], fmt)
                    ax.text(j, i, "0.0" if txt in ("-0.0", "0.0") and fmt == ".1f" else txt, ha="center",
                            va="center", fontsize=fontsize * 0.85, color=color)
        cb = fig.colorbar(im, ax=ax, shrink=0.85)
        cb.set_label(cbar_label, fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize * 0.85)
        if note:
            ax.text(0, 1.02, note, transform=ax.transAxes, ha="left", va="bottom", fontsize=fontsize * 0.7, color="0.4")
        fig.tight_layout()
        if stamp_source:
            _stamp(fig, stamp_source)
    return fig


def plot_spectrum_evolution(result, name, fontsize=FIG8_FONTSIZE, stamp=False):
    import matplotlib.pyplot as plt
    from scipy.signal import savgol_filter
    sp = result["spectra"][name]
    flows = result["df"].set_index("filename")[QCOLS].loc[name]
    A = np.asarray(sp["abs"]); wl = np.asarray(sp["wl"]); t = np.asarray(sp["t"])
    wl_range = EVOL_WL_RANGE
    m = (wl >= wl_range[0]) & (wl <= wl_range[1]); wlm = wl[m]
    win = min(SG_WINDOW, len(wlm) - (len(wlm) + 1) % 2)
    if win <= SG_POLY:
        win = SG_POLY + 1 + (SG_POLY % 2)
    norm = mpl.colors.Normalize(t.min(), t.max())
    with plt.rc_context(PUB_STYLE):
        fig, ax = plt.subplots(figsize=EVOL_FIGSIZE)
        ymax = 0.0
        for j in np.argsort(t):
            s = savgol_filter(A[j][m], win, SG_POLY)
            ax.plot(wlm, s, color=mpl.colormaps[EVOL_CMAP](norm(t[j])), lw=0.9, alpha=0.9)
            ymax = max(ymax, s.max())
        ax.set_xlim(*wl_range); ax.set_ylim(0, ymax * 1.03)
        ax.set_xlabel("Wavelength (nm)", fontsize=fontsize)
        ax.set_ylabel("Extinction (a.u.)", fontsize=fontsize)
        _frame_axes(ax, fontsize)
        env = savgol_filter(A[:, m].max(0), win, SG_POLY)
        left = wlm[np.argmax(env)] > 0.5 * (wl_range[0] + wl_range[1])
        txt = "\n".join(f"{REAGENTS[k]:>7s} {flows[QCOLS[k]]:4.2f}" for k in range(5)) + "\n  (µL/s)"
        ax.text(0.035 if left else 0.965, 0.965, txt, transform=ax.transAxes, ha="left" if left else "right",
                va="top", fontsize=fontsize * EVOL_FLOWBOX_SCALE, family="monospace",
                bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=EVOL_CMAP); sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.035)
        cb.set_label("time (s)", fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize * 0.85, direction="in")
        cb.outline.set_linewidth(0.9)
        fig.tight_layout()
        if stamp:
            _stamp(fig, "10_eda_S14.ipynb")
    return fig


def plot_kinetics(result, name, stamp=False):
    """Ext_max(t) (left axis) and spectral area(t) (right axis), logistic fits, max-rate box."""
    import matplotlib.pyplot as plt
    k = result["kinetics"][name]
    t, y1, y2 = k["t"], k["ext_max"], k["area"]
    c1, c2 = "#0072B2", "#009E73"
    note = None
    if k["rates"] is not None:
        note = (f"max rate:\n  Ext$_{{max}}$ {k['rates']['ext_max']:.3f} /s\n"
                f"  area {k['rates']['area']:.1f} (a.u.·nm)/s")
    with plt.rc_context(PUB_STYLE):
        fig, ax1 = plt.subplots(figsize=FIG10E_FIGSIZE)
        m1 = np.isfinite(y1)
        ax1.scatter(t[m1], y1[m1], s=20, color=c1, alpha=0.45, zorder=2)
        if k["fit_ext_max"][0] is not None:
            ax1.plot(k["fit_ext_max"][0], k["fit_ext_max"][1], color=c1, lw=2, zorder=3)
        ax1.set_xlabel("Time (s)", fontsize=16, fontweight="bold")
        ax1.set_ylabel(r"$Ext_{\max}$ (a.u.)", fontsize=16, fontweight="bold", color=c1)
        ax1.tick_params(axis="x", labelsize=14); ax1.tick_params(axis="y", colors=c1, labelsize=14)
        ax1.spines["left"].set_color(c1)
        ax2 = ax1.twinx()
        m2 = np.isfinite(y2)
        ax2.scatter(t[m2], y2[m2], s=20, color=c2, alpha=0.45, zorder=2)
        if k["fit_area"][0] is not None:
            ax2.plot(k["fit_area"][0], k["fit_area"][1], color=c2, lw=2, zorder=3)
        ax2.set_ylabel("Spectral area (a.u.·nm)", fontsize=16, fontweight="bold", color=c2)
        ax2.tick_params(axis="y", colors=c2, labelsize=14)
        ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(True)
        ax2.spines["right"].set_color(c2); ax2.spines["right"].set_linewidth(1.5)
        if note:
            ax1.text(0.03, 0.96, note, transform=ax1.transAxes, ha="left", va="top", fontsize=13,
                     bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85))
        ax1.set_title(_short(name), fontsize=16, fontweight="bold")
        fig.tight_layout()
        if stamp:
            _stamp(fig, "10_eda_S14.ipynb")
    return fig


### `figS08`

In [ ]:
"""Figure S8 — coverage of the 5-pump simplex by the 100 LHS conditions, coloured by λmax
(original: 2FAST_ML/10_eda_S14.ipynb, cell `fig5_coverage`, saved as figures_eda/fig5_simplex_coverage).

`title=True, stamp=True` reproduces the original PNG exactly; the SI version has neither.
"""
import matplotlib as mpl
import matplotlib.pyplot as plt


DATASET = "csv_romain"


def compute_coverage(raw=None):
    return table(SETTINGS_NB10, raw)[0]


def plot(df, title=False, stamp=False):
    with plt.rc_context(PUB_STYLE):
        fig, axes = plt.subplots(5, 5, figsize=(11, 11))
        norm = mpl.colors.Normalize(df.lambda_max.min(), df.lambda_max.max())
        cmap = mpl.cm.viridis
        for i in range(5):
            for j in range(5):
                ax = axes[i, j]
                if i == j:
                    ax.hist(df[QCOLS[i]], bins=14, color="#888888"); ax.set_yticks([])
                else:
                    ax.scatter(df[QCOLS[j]], df[QCOLS[i]], c=df.lambda_max, cmap=cmap, norm=norm, s=14,
                               alpha=0.8, edgecolor="white", linewidth=0.2)
                if i == 4: ax.set_xlabel(f"{QCOLS[j]}\n{REAGENTS[j]}", fontsize=9)
                else:      ax.set_xticklabels([])
                if j == 0: ax.set_ylabel(f"{QCOLS[i]}\n{REAGENTS[i]}", fontsize=9)
                else:      ax.set_yticklabels([])
                ax.tick_params(labelsize=7)
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        cb = fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.02)
        cb.set_label(r"$\lambda_{\max}$ (nm)")
        if title:
            fig.suptitle(f"{DATASET}: simplex coverage of the 5-pump space (sum Q = 14 uL/s)",
                         y=0.92, fontsize=13, fontweight="bold")
        if stamp:
            _stamp(fig, "10_eda_S14.ipynb")
    return fig


### `figS12`

In [ ]:
"""Figure S12 — PCA of the endpoint spectra of the 100 LHS conditions (88 non-flat), 6 components, max-normalised:
loadings of PC1-PC4 (+ mean spectrum) and the PC1-PC2 score space coloured by λmax.

Port of 2FAST_ML/PCA/pca_decomposition.ipynb (figures_pca/fig3_components, fig6_score_space); same code and
parameters. λmax here = raw argmax of the endpoint spectrum (as in that notebook), not the instrument value.
`stamp=True` reproduces the original PNGs (footer "pca_decomposition.ipynb").
"""
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from matplotlib.colors import Normalize


MIN_WL, MAX_WL = 310.0, 1000.0
N_COMPONENTS = 6
NORMALIZE = "max"
FLAT_THRESHOLD = 0.015
STYLE = {"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11,
         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25}
ABS_LABEL = "Absorbance" if NORMALIZE == "none" else "Normalized absorbance"


def normalize_vec(y, mode):
    if mode == "none":
        return y
    if mode == "max":
        p = float(np.max(np.abs(y)))
        return y / p if p > 1e-12 else y
    if mode == "area":
        a = float(np.trapz(np.abs(y)))
        return y / a if a > 1e-12 else y
    raise ValueError(mode)


def fit_pca(spectra, wl, n_components, normalize="none"):
    X = np.vstack([normalize_vec(r, normalize) for r in np.atleast_2d(spectra)])
    mean = X.mean(axis=0)
    U, S, Vt = np.linalg.svd(X - mean, full_matrices=False)
    var = (S ** 2) / (X.shape[0] - 1)
    evr = var / var.sum()
    return {"wl": np.asarray(wl, float), "mean": mean, "comps": Vt[:n_components],
            "explained_variance": var[:n_components], "explained_variance_ratio": evr[:n_components],
            "normalize": normalize, "n_components": n_components}


def project(model, s):
    return model["comps"] @ (normalize_vec(np.asarray(s, float), model["normalize"]) - model["mean"])


def load_endpoints(raw=None):
    """Endpoint spectra in the original order (sorted file names of the csv_romain flat copy)."""
    raw = RAW if raw is None else raw
    items = sorted(file_list(), key=lambda r: os.path.basename(r["raw_path"]))
    flows, spectra, names, wl = [], [], [], None
    for it in items:
        with h5py.File(raw / it["raw_path"], "r") as f:
            x = np.asarray(f["conditions/flow_set"][:], float)
            w = np.asarray(f["spectra/wavelength"][:], float)
            a = f["result/extinction_final"][:] if "result/extinction_final" in f else f["spectra/extinction"][-1]
            a = np.asarray(a, float)
        if wl is None:
            wl = w
        elif w.shape != wl.shape or not np.allclose(w, wl):
            raise ValueError(f"different wavelength grid: {it['raw_path']}")
        flows.append(x); spectra.append(a); names.append(it["filename"])
    return np.array(flows), np.array(spectra), wl, names


def compute_pca(raw=None):
    flows, spectra, wl, names = load_endpoints(raw)
    keep = (wl >= MIN_WL) & (wl <= MAX_WL)
    wl, spectra = wl[keep], spectra[:, keep]
    lam_max = wl[np.argmax(spectra, axis=1)]
    m = (wl >= MIN_WL) & (wl <= MAX_WL)
    flat = spectra[:, m].std(axis=1) < FLAT_THRESHOLD
    spectra_fit, lam_fit = spectra[~flat], lam_max[~flat]
    pca = fit_pca(spectra_fit, wl, N_COMPONENTS, NORMALIZE)
    scores = np.vstack([project(pca, s) for s in spectra_fit])
    return dict(wl=wl, flat=flat, names=names, spectra_fit=spectra_fit, lam_fit=lam_fit, pca=pca, scores=scores)


def _stamp_pca(fig):
    fig.text(0.995, 0.005, "pca_decomposition.ipynb", ha="right", va="bottom", fontsize=6, color="0.6")


def save_pca(fig, path):
    with plt.rc_context(STYLE):
        fig.savefig(path)


def plot_components(r, stamp=False):
    wl, pca = r["wl"], r["pca"]; ratio = pca["explained_variance_ratio"]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
        ax[0].plot(wl, pca["mean"], color="k", lw=1.5)
        ax[0].set_title("Mean spectrum", fontsize=11)
        ax[0].set_xlabel("Wavelength (nm)"); ax[0].set_ylabel(ABS_LABEL); ax[0].set_xlim(MIN_WL, MAX_WL)
        colors = cm.tab10(np.linspace(0, 1, 10))
        for j in range(min(4, N_COMPONENTS)):
            ax[1].plot(wl, pca["comps"][j], lw=1.3, color=colors[j], label=f"PC{j+1} ({ratio[j]*100:.1f}%)")
        ax[1].axhline(0, color="0.7", lw=0.8)
        ax[1].set_title("Principal components (spectral modes)", fontsize=11)
        ax[1].set_xlabel("Wavelength (nm)"); ax[1].set_ylabel("Loading"); ax[1].set_xlim(MIN_WL, MAX_WL)
        ax[1].legend(frameon=False, fontsize=9)
        if stamp:
            _stamp_pca(fig)
    return fig


def plot_scores(r, stamp=False):
    ratio = r["pca"]["explained_variance_ratio"]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(6.5, 5))
        sc = ax.scatter(r["scores"][:, 0], r["scores"][:, 1], c=r["lam_fit"], cmap="viridis",
                        s=30, edgecolor="k", linewidths=0.3)
        fig.colorbar(sc, ax=ax, label=r"$\lambda_{max}$ (nm)")
        ax.set_xlabel(f"PC1 ({ratio[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({ratio[1]*100:.1f}%)")
        if stamp:
            _stamp_pca(fig)
    return fig


## Figure

In [ ]:
from IPython.display import Image, display
import numpy as np
FIGURES.mkdir(exist_ok=True)

df = compute_coverage()
save(plot(df), FIGURES / 'figS08_simplex_coverage.png')
save(plot(df), FIGURES / 'figS08_simplex_coverage.pdf')
display(Image(str(FIGURES / 'figS08_simplex_coverage.png')))


In [ ]:
r = compute_pca()
evr = r['pca']['explained_variance_ratio']
print(f"flat removed: {int(r['flat'].sum())} | fitted: {len(r['spectra_fit'])}")
for k, (e, c) in enumerate(zip(evr, np.cumsum(evr)), 1):
    print(f'PC{k}: {100*e:5.1f}%   cumulative {100*c:5.1f}%')
for name, fig in (('figS12_scores', plot_scores(r)), ('figS12_loadings', plot_components(r))):
    for ext in ('png', 'pdf'):
        save_pca(fig, FIGURES / f'{name}.{ext}')
    display(Image(str(FIGURES / f'{name}.png')))
